In [1]:
import os
import re
import sys
import logging
import json
import numpy as np
import pandas as pd
import pickle
import xgboost as xgb
import plotly.graph_objects as go
from collections import defaultdict
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

from typing import Optional
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
import joblib
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sqlalchemy import create_engine
sys.path.append("/dss/work/rirg2545/actionable-hypotension/")



# ----------------------
# Configuration
# ----------------------
MODELS_DIR = "/dss/work/rirg2545/actionable-hypotension/models_given/calibrated"
SUBANALYSIS_DIR = "/dss/work/rirg2545/actionable-hypotension/extended_evaluation_review/results"
LOG_LEVEL = logging.INFO
# ----------------------
# Set up logging
# ----------------------
logging.basicConfig(
level=LOG_LEVEL,
format="%(asctime)s [%(levelname)s] %(message)s",
datefmt="%Y-%m-%d %H:%M:%S")
# -----
# DB connection
DATABASE_URI = "postgresql+psycopg2://rirg2545@localhost:5434/mimic"
engine = create_engine(DATABASE_URI, future=True)

decision_thresholds = {
"mix": 0.0011452179169282317,
"inv_model": 0.0020985996816307306,
"noninv_model": 0.0008126477478072047}

UNCALIBRATED_MODELS_DIR = "/dss/work/rirg2545/actionable-hypotension/models_given/uncalibrated"
CALIBRATED_MODELS_DIR = "/dss/work/rirg2545/actionable-hypotension/extended_evaluation_review/models_calibrated_unbundled"

CREATE TEST FULLS

In [2]:
def load_calibrated_model_predictions(
    model_name: str,
    x_val: pd.DataFrame,
    x_test: pd.DataFrame
):
    """
    Loads XGBoost model (JSON) + calibrator (pkl) and returns calibrated predictions.

    Returns:
        Tuple[np.ndarray, np.ndarray, Tuple[Booster, Calibrator]]:
            - y_val_pred (calibrated probabilities)
            - y_test_pred (calibrated probabilities)
            - (booster, calibrator)
    """
    import xgboost as xgb

    # --- Load model ---
    model_path = os.path.join(UNCALIBRATED_MODELS_DIR, f"{model_name}.json")
    logging.info(f"Lade Modell von: {model_path}")

    booster = xgb.Booster()
    booster.load_model(model_path)

    # --- Load calibrator ---
    calibrator_path = os.path.join(
        CALIBRATED_MODELS_DIR, f"{model_name}_calibrator.pkl"
    )
    logging.info(f"Lade Kalibrator von: {calibrator_path}")

    calibrator = joblib.load(calibrator_path)

    # --- Predict raw ---
    dval = xgb.DMatrix(x_val)
    dtest = xgb.DMatrix(x_test)

    y_val_raw = booster.predict(dval)
    y_test_raw = booster.predict(dtest)

    # --- Apply calibration ---
    y_val_pred = calibrator.transform(y_val_raw)
    y_test_pred = calibrator.transform(y_test_raw)

    return y_val_pred, y_test_pred, (booster, calibrator)

In [3]:
def load_and_prepare_data_xgb(table_name):
    
    cache = {} 
    if table_name in cache:
        logging.info(f"🔁 Using cached data for {table_name} ")
        return cache[table_name]

    logging.info(f" Loading val/test data from evaluation.{table_name}")
    

    df = pd.read_sql(f"""
        SELECT * FROM evaluation.{table_name}
        WHERE split IN ('val', 'test')
        ORDER BY subject_id, icustay_id, context_start
    """, engine)

    # Vorverarbeitung
    # IDs und Kontext separat sichern
    id_cols = ["subject_id", "icustay_id", "context_start", "context_end", "currently_on_cat", "cat_label"]
    id_df = df[id_cols + ["split"]].copy()

    # Label engineering
    df["label"] = df["positive_event"].astype(int)

    excluded = {"positive_event", "positive_sample", "split", "label", "treatment_given", "only_2_values"} | set(id_cols)
    
    logging.info(f"Features excluded: {excluded}")

    feature_cols = [c for c in df.columns if c not in excluded]

    val_df = df[df["split"] == "val"]
    test_df = df[df["split"] == "test"]

    x_val = val_df[feature_cols]
    y_val = val_df["label"]
    x_test = test_df[feature_cols]
    y_test = test_df["label"]

    # Reihenfolge:
    # Die IDs passend zum Split filtern
    id_val = id_df[id_df["split"] == "val"].reset_index(drop=True)
    id_test = id_df[id_df["split"] == "test"].reset_index(drop=True)

    logging.info(f"✅ Data loaded and cached for {table_name}")

    cache[table_name] = (x_val, y_val, x_test, y_test, feature_cols, id_val, id_test)
    return cache[table_name]

In [4]:
def create_test_full_for_table(table_name: str):
    """
    Erstellt ein DataFrame `test_full` mit Features, Label und Modellvorhersage.
    Zusätzliche features für Subgruppenanalyse: 
    """

    # Modellname und Tabellennamen anpassen
    model_name = f"xgb_{table_name}"
    table_name = f"merged_{table_name}_features_w_cat_status_w_label" # updated to include the subgroup-info "currently on cat yes or no"

    # Lade Daten einmal – inkl. y-Test und den indices für val und test
    x_val, y_val, x_test, y_test, feature_cols, id_val, id_test = load_and_prepare_data_xgb(table_name)

    # Lade Modell und Vorhersagen machen

    y_val_pred, y_test_pred, model = load_calibrated_model_predictions(model_name, x_val, x_test)

    # Erstelle Test-Full-DataFrame
    test_full = x_test.copy()
    test_full["label"] = y_test.values
    test_full["preds"] = y_test_pred

    # Jetzt: concat id_test (Identifikatoren und context_start) mit test_full (Features, Label, Preds)
    test_full = pd.concat([id_test.reset_index(drop=True), test_full.reset_index(drop=True)], axis=1)

    #### Mehrere Zusatzinformationen für die Subgruppenanalyse aus der originalen tabelle holen + neue erstellen ####

    # >>>>> Extended_Evaluation_Columns <<<<<
    # updated to include each individual exclusion criteria
    extended_evaluation_df = pd.read_sql(f"""
    SELECT f.icustay_id, f.subject_id, f.transplant, f.stroke,f.brain_injury,f.brain_hemorrhage, f.exclusion, f.age, f.saps_score, f.sapsii_score, f.sofa_score, f.icu_type, f.heart_surgery_patient
    FROM evaluation.fused_subgroups_mv_excluded_unpacked f
    JOIN ce_approach.split_all_subjects s ON f.subject_id = s.subject_id
    WHERE s.split = 'test'
""", engine)
    
    
    # Merge der Zusatzinfos: exclusion,age etc.)
    test_full = test_full.merge(extended_evaluation_df, on=["icustay_id", "subject_id"], how="left")
    

    return test_full

In [5]:
def create_all_test_fulls(table_names: list[str]) -> dict[str, pd.DataFrame]:
    """
    Erstellt test_full DataFrames für alle angegebenen Tabellen.
    
    Returns:
        dict mapping model name (e.g. 'mix_model') to its test_full DataFrame
    """
    dfs = {}
    for table_name in table_names:
        dfs[table_name] = create_test_full_for_table(table_name)
    return dfs

In [6]:
def bootstrap_auc(y_true, y_pred, n_bootstrap=1000, ci=0.95, seed=42):
    """
    Returns (auc, lower, upper) using percentile bootstrap.
    """
    rng = np.random.default_rng(seed)
    aucs = []
    n = len(y_true)
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        y_t = y_true_arr[idx]
        y_p = y_pred_arr[idx]
        if len(np.unique(y_t)) < 2:
            continue  # skip degenerate bootstrap samples
        aucs.append(roc_auc_score(y_t, y_p))

    alpha = (1 - ci) / 2
    lower = np.percentile(aucs, 100 * alpha)
    upper = np.percentile(aucs, 100 * (1 - alpha))
    point = roc_auc_score(y_true_arr, y_pred_arr)
    return point, lower, upper

## Evaluate Subgroups 

In [14]:
def build_subgroup_configs(df: pd.DataFrame) -> list[dict]:
    """
    Define all subgroups as filter configurations.
    Each entry: name, filter_condition, and optional metadata.
    """
    configs = [
        
        # --- last MAP ---
        {"name": "last < 65",
         "filter": lambda df: df["last"] <= 65},
        {"name": "last 70-80",
         "filter": lambda df: (df["last"] > 65) & (df["last"] <= 70)},
        {"name": "last 80-85",
         "filter": lambda df: (df["last"] > 70) & (df["last"] <= 100)},
        {"name": "last > 85",
         "filter": lambda df: df["last"] > 100},
         
        
    ]
    return configs

In [8]:
def build_subgroup_configs(df: pd.DataFrame) -> list[dict]:
    """
    Define all subgroups as filter configurations.
    Each entry: name, filter_condition, and optional metadata.
    """
    configs = [
        
        # --- Baseline ---
        {"name": "All patients",
         "filter": None},
         # --- Catecholamine subgroup analyses ---
        {"name": "No catecholamine event",
        "filter": lambda df: df["cat_label"] == "negative"},

        {"name": "All vasopressors (any catecholamine)",
        "filter": lambda df: df["cat_label"] != "negative"},

        {"name": "Norepinephrine events",
        "filter": lambda df: df["cat_label"].isin(["Norepinephrine", "negative"])},

        {"name": "Vasopressin events",
        "filter": lambda df: df["cat_label"].isin(["Vasopressin", "negative"])},

        {"name": "Epinephrine events",
        "filter": lambda df: df["cat_label"].isin(["Epinephrine", "negative"])},

        {"name": "Dopamine events",
        "filter": lambda df: df["cat_label"].isin(["Dopamine", "negative"])},

        {"name": "Dobutamine events",
        "filter": lambda df: df["cat_label"].isin(["Dobutamine", "negative"])},

        {"name": "Phenylephrine events",
        "filter": lambda df: df["cat_label"].isin(["Phenylephrine", "negative"])},

        {"name": "Milrinone events",
        "filter": lambda df: df["cat_label"].isin(["Milrinone", "negative"])},

        {"name": "Vasopressor-dominant agents",
        "filter": lambda df: df["cat_label"].isin(["Norepinephrine", "Phenylephrine", "Vasopressin", "negative"])},

        {"name": "Inotrope-dominant agents",
        "filter": lambda df: df["cat_label"].isin(["Dobutamine", "Milrinone","negative"])},

        {"name": "Mixes-action agents",
        "filter": lambda df: df["cat_label"].isin(["Dopamine", "Epinephrine", "negative"])},
        # --- Exclusion ---
        {"name": "Excluded patients_all",
         "filter": lambda df: df["exclusion"] == 1},
        {"name": "Non-excluded patients",
         "filter": lambda df: df["exclusion"] == 0},
          # --- Clinical subgroups ---
        {"name": "Transplant patients",
         "filter": lambda df: df["transplant"] == 1},
        {"name": "Non-transplant patients",
         "filter": lambda df: df["transplant"] == 0},
        {"name": "Stroke patients",
         "filter": lambda df: df["stroke"] == 1},
        {"name": "Non-stroke patients",
         "filter": lambda df: df["stroke"] == 0},
        {"name": "Brain injury patients",
         "filter": lambda df: df["brain_injury"] == 1},
        {"name": "Non-brain injury patients",
         "filter": lambda df: df["brain_injury"] == 0},
        {"name": "Brain hemorrhage patients",
         "filter": lambda df: df["brain_hemorrhage"] == 1},
        {"name": "Non-brain hemorrhage patients",
         "filter": lambda df: df["brain_hemorrhage"] == 0},

         # --- currently on catecholamines ---
        {"name": "On Catecholamines",
         "filter": lambda df: df["currently_on_cat"] == 1},
        {"name": "Not on catecholamines",
         "filter": lambda df: df["currently_on_cat"] == 0},

        # --- Age ---
        {"name": "Age < 70",
         "filter": lambda df: df["age"] < 70},
        {"name": "Age 70-80",
         "filter": lambda df: (df["age"] >= 70) & (df["age"] < 80)},
        {"name": "Age 80-85",
         "filter": lambda df: (df["age"] >= 80) & (df["age"] < 85)},
        {"name": "Age > 85",
         "filter": lambda df: df["age"] > 85},

        # --- ICU type ---
        {"name": "Surgical ICU",
         "filter": lambda df: df["icu_type"] == "surgical"},
        {"name": "Non-surgical ICU",
         "filter": lambda df: df["icu_type"] == "non-surgical"},

        # --- Heart surgery ---
        {"name": "Heart surgery",
         "filter": lambda df: df["heart_surgery_patient"] == 1},
        {"name": "No heart surgery",
         "filter": lambda df: df["heart_surgery_patient"] == 0},

           # --- SOFA clinical cutoffs ---
        {"name": "SOFA < 6 (low severity)",
         "filter": lambda df: df["sofa_score"] < 6},
        {"name": "SOFA 6-10 (moderate severity)",
         "filter": lambda df: (df["sofa_score"] >= 6) & (df["sofa_score"] <= 10)},
        {"name": "SOFA >= 11 (high severity)",
         "filter": lambda df: df["sofa_score"] >= 11},

        # --- SOFA tertiles (data-driven) ---
        {"name": "SOFA low (tertile 1)",
         "filter": lambda df: df["sofa_score"] <= df["sofa_score"].quantile(0.33)},
        {"name": "SOFA mid (tertile 2)",
         "filter": lambda df: (df["sofa_score"] > df["sofa_score"].quantile(0.33)) &
                              (df["sofa_score"] <= df["sofa_score"].quantile(0.66))},
        {"name": "SOFA high (tertile 3)",
         "filter": lambda df: df["sofa_score"] > df["sofa_score"].quantile(0.66)},

        # --- SAPS I clinical cutoffs (range 0-56) ---
        {"name": "SAPS I < 14 (low severity)",
         "filter": lambda df: df["saps_score"] < 14},
        {"name": "SAPS I 14-24 (moderate severity)",
         "filter": lambda df: (df["saps_score"] >= 14) & (df["saps_score"] <= 24)},
        {"name": "SAPS I >= 25 (high severity)",
         "filter": lambda df: df["saps_score"] >= 25},

        # --- SAPS I tertiles (data-driven) ---
        {"name": "SAPS I low (tertile 1)",
         "filter": lambda df: df["saps_score"] <= df["saps_score"].quantile(0.33)},
        {"name": "SAPS I mid (tertile 2)",
         "filter": lambda df: (df["saps_score"] > df["saps_score"].quantile(0.33)) &
                              (df["saps_score"] <= df["saps_score"].quantile(0.66))},
        {"name": "SAPS I high (tertile 3)",
         "filter": lambda df: df["saps_score"] > df["saps_score"].quantile(0.66)},

        # --- SAPS II clinical cutoffs (range 0-163)---
        {"name": "SAPS II < 40 (low severity)",
         "filter": lambda df: df["sapsii_score"] < 40},
        {"name": "SAPS II 40-51 (moderate severity)",
         "filter": lambda df: (df["sapsii_score"] >= 40) & (df["sapsii_score"] <= 51)},
        {"name": "SAPS II >= 52 (high severity)",
         "filter": lambda df: df["sapsii_score"] >= 52},

        # --- SAPS II tertiles (data-driven) ---
        {"name": "SAPS II low (tertile 1)",
         "filter": lambda df: df["sapsii_score"] <= df["sapsii_score"].quantile(0.33)},
        {"name": "SAPS II mid (tertile 2)",
         "filter": lambda df: (df["sapsii_score"] > df["sapsii_score"].quantile(0.33)) &
                              (df["sapsii_score"] <= df["sapsii_score"].quantile(0.66))},
        {"name": "SAPS II high (tertile 3)",
         "filter": lambda df: df["sapsii_score"] > df["sapsii_score"].quantile(0.66)}

        
    ]
    return configs

In [9]:
def evaluate_subgroup(
    df: pd.DataFrame,
    filter_condition=None,
    label_col: str = "label",
    pred_col: str = "preds",
    threshold_value: int = 0,
    model_name: str = "Unnamed Model",
    name: str = "Unnamed Subgroup",
    n_bootstrap: int = 1000,
    ci: float = 0.95,
    min_samples: int = 30,
    verbose: bool = True
) -> dict | None:
    """
    Evaluates a subgroup of the test set.
    Returns AUC with bootstrapped CI, sensitivity, specificity.
    """

    # --- Apply filter ---
    if filter_condition is None:
        subset = df
    elif callable(filter_condition):
        subset = df[filter_condition(df)]
    else:
        subset = df[filter_condition]

    if len(subset) < min_samples:
        if verbose:
            print(f"Skipping '{name}' (n={len(subset)}) - too small.")
        return None

    y_true = subset[label_col]
    y_pred = subset[pred_col]

    if len(np.unique(y_true)) < 2:
        if verbose:
            print(f"Skipping '{name}' - only one class present.")
        return None

    # --- Bootstrapped AUC ---
    auc, lower, upper = bootstrap_auc(
        y_true, y_pred, n_bootstrap=n_bootstrap, ci=ci
    )

    # --- Get threshold for this model ---
    if threshold_value is None:
        raise ValueError("threshold must be provided.")

    # --- Sensitivity / Specificity ---
    y_pred_label = (y_pred >= threshold_value).astype(int)
    cm = confusion_matrix(y_true, y_pred_label)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    n_pos = int(y_true.sum())
    n_total = len(y_true)

    if verbose:
        print(f"[{model_name}] {name}")
        print(f"  → N = {n_total} | N_pos = {n_pos}")
        print(f"  → AUC = {auc:.3f} [{lower:.3f}–{upper:.3f}]")
        print(f"  → Threshold = {threshold_value:.6f}")
        print(f"  → Sensitivity = {sensitivity:.3f} | Specificity = {specificity:.3f}")

    return {
        "auc": auc,
        "auc_lower": lower,
        "auc_upper": upper,
        "n_pos": n_pos,
        "n_total": n_total,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "threshold": threshold_value
    }

In [10]:
def evaluate_all_subgroups_latex_ready(
    
    threshold_dict: dict,
    min_samples: int = 30,
    n_bootstrap: int = 1000,
    ci: float = 0.95,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Config-driven subgroup evaluation across all models.
    Calls evaluate_subgroup per config entry.
    Output: one row per subgroup, one column per model.
    Cell format: AUC [lower–upper], n_pos/n_total, sens, spec
    """
    dfs = create_all_test_fulls(["mix"])
    subgroup_rows = defaultdict(dict)

    for model_name, df in dfs.items():
        threshold = threshold_dict.get(model_name)
        if threshold is None:
            raise ValueError(f"Threshold für Modell '{model_name}' nicht gefunden.")

        configs = build_subgroup_configs(df)

        for cfg in configs:
            name = cfg["name"]
            result = evaluate_subgroup(
                df=df,
                filter_condition=cfg["filter"],
                label_col="label",
                pred_col="preds",
                threshold_value=threshold,
                model_name=model_name,
                name=name,
                n_bootstrap=n_bootstrap,
                ci=ci,
                min_samples=min_samples,
                verbose=verbose
            )

            if result is None:
                continue

            subgroup_rows[name][model_name] = (
                f"{result['auc']:.3f} [{result['auc_lower']:.3f}–{result['auc_upper']:.3f}], "
                f"{result['n_pos']}/{result['n_total']}, "
                f"{result['sensitivity']:.3f}, "
                f"{result['specificity']:.3f}"
            )

    # Assemble latex-ready DataFrame
    rows = []
    for name, model_entries in subgroup_rows.items():
        row = {"Subgroup": name}
        for model_name in dfs.keys():
            row[model_name] = model_entries.get(model_name, "—")
        rows.append(row)

    return pd.DataFrame(rows).set_index("Subgroup")

Aufruf 


In [11]:

results = evaluate_all_subgroups_latex_ready(
    threshold_dict=decision_thresholds,
    min_samples = 30,
    n_bootstrap = 1000,
    ci = 0.95,
    verbose = True) 

    
#print(type(results))

#results.to_csv(os.path.join(SUBANALYSIS_DIR, "results_subgroup_analysis_newest_2.csv"), index=False)


2026-04-27 12:54:15 [INFO]  Loading val/test data from evaluation.merged_mix_features_w_cat_status_w_label
2026-04-27 12:54:43 [INFO] Features excluded: {'currently_on_cat', 'icustay_id', 'subject_id', 'only_2_values', 'treatment_given', 'label', 'context_start', 'cat_label', 'context_end', 'positive_sample', 'positive_event', 'split'}
2026-04-27 12:54:43 [INFO] ✅ Data loaded and cached for merged_mix_features_w_cat_status_w_label
2026-04-27 12:54:43 [INFO] Lade Modell von: /dss/work/rirg2545/actionable-hypotension/models_given/uncalibrated/xgb_mix.json
2026-04-27 12:54:44 [INFO] Lade Kalibrator von: /dss/work/rirg2545/actionable-hypotension/extended_evaluation_review/models_calibrated_unbundled/xgb_mix_calibrator.pkl


[mix] All patients
  → N = 1116174 | N_pos = 2256
  → AUC = 0.823 [0.814–0.831]
  → Threshold = 0.001145
  → Sensitivity = 0.853 | Specificity = 0.612
Skipping 'No catecholamine event' - only one class present.
Skipping 'All vasopressors (any catecholamine)' - only one class present.
[mix] Norepinephrine events
  → N = 1114640 | N_pos = 722
  → AUC = 0.835 [0.821–0.849]
  → Threshold = 0.001145
  → Sensitivity = 0.871 | Specificity = 0.612
[mix] Vasopressin events
  → N = 1114106 | N_pos = 188
  → AUC = 0.859 [0.833–0.882]
  → Threshold = 0.001145
  → Sensitivity = 0.915 | Specificity = 0.612
[mix] Epinephrine events
  → N = 1113980 | N_pos = 62
  → AUC = 0.913 [0.881–0.942]
  → Threshold = 0.001145
  → Sensitivity = 0.968 | Specificity = 0.612
[mix] Dopamine events
  → N = 1114020 | N_pos = 102
  → AUC = 0.850 [0.814–0.882]
  → Threshold = 0.001145
  → Sensitivity = 0.882 | Specificity = 0.612
[mix] Dobutamine events
  → N = 1113950 | N_pos = 32
  → AUC = 0.871 [0.806–0.919]
  → Thres

In [15]:
results = evaluate_all_subgroups_latex_ready(
    threshold_dict=decision_thresholds,
    min_samples = 30,
    n_bootstrap = 1000,
    ci = 0.95,
    verbose = True) 

results.to_csv(os.path.join(SUBANALYSIS_DIR, "extended_subgroup_analysis_v5.csv"), index=True)

# - Decomposition of endpoint
# unit context
# icu type
# heart surgery yes no 
# -age
# severity (sofa, saps 2)
# - special/excluded patients:
# transplant
# stroke
# brain injury 
# brain hemorhhage 
# exclusion_all



2026-04-27 14:00:02 [INFO]  Loading val/test data from evaluation.merged_mix_features_w_cat_status_w_label
2026-04-27 14:00:34 [INFO] Features excluded: {'currently_on_cat', 'icustay_id', 'subject_id', 'only_2_values', 'treatment_given', 'label', 'context_start', 'cat_label', 'context_end', 'positive_sample', 'positive_event', 'split'}
2026-04-27 14:00:35 [INFO] ✅ Data loaded and cached for merged_mix_features_w_cat_status_w_label
2026-04-27 14:00:35 [INFO] Lade Modell von: /dss/work/rirg2545/actionable-hypotension/models_given/uncalibrated/xgb_mix.json
2026-04-27 14:00:35 [INFO] Lade Kalibrator von: /dss/work/rirg2545/actionable-hypotension/extended_evaluation_review/models_calibrated_unbundled/xgb_mix_calibrator.pkl


[mix] last < 65
  → N = 228988 | N_pos = 1069
  → AUC = 0.778 [0.764–0.792]
  → Threshold = 0.001145
  → Sensitivity = 0.968 | Specificity = 0.207
[mix] last 70-80
  → N = 147174 | N_pos = 325
  → AUC = 0.754 [0.729–0.777]
  → Threshold = 0.001145
  → Sensitivity = 0.874 | Specificity = 0.462
[mix] last 80-85
  → N = 649329 | N_pos = 763
  → AUC = 0.808 [0.792–0.823]
  → Threshold = 0.001145
  → Sensitivity = 0.713 | Specificity = 0.758
[mix] last > 85
  → N = 90683 | N_pos = 99
  → AUC = 0.797 [0.747–0.844]
  → Threshold = 0.001145
  → Sensitivity = 0.626 | Specificity = 0.826


In [19]:
df = pd.read_csv(
    os.path.join(SUBANALYSIS_DIR, "extended_subgroup_analysis_v3.csv"),
    index_col=0
)

df

,AUROC [CI],pos/total,Sensitivity,Specificity
Subgroup,,,,
All patients,0.823 [0.814–0.831],2256/1116174,0.853,0.612
Norepinephrine events,0.835 [0.821–0.849],722/1114640,0.871,0.612
Vasopressin events,0.859 [0.833–0.882],188/1114106,0.915,0.612
Epinephrine events,0.913 [0.881–0.942],62/1113980,0.968,0.612
Dopamine events,0.850 [0.814–0.882],102/1114020,0.882,0.612
Dobutamine events,0.871 [0.806–0.919],32/1113950,0.906,0.612
Phenylephrine events,0.801 [0.789–0.813],1094/1115012,0.821,0.612
Milrinone events,0.792 [0.742–0.841],56/1113974,0.839,0.612
Vasopressor-dominant agents,0.819 [0.809–0.827],2004/1115922,0.848,0.612


In [13]:
df = pd.read_csv(
    os.path.join(SUBANALYSIS_DIR, "extended_subgroup_analysis_v2.csv"),
    index_col=0
)

df

,AUROC [CI],pos/total,Sensitivity,Specificity
Subgroup,,,,
All patients,0.822 [0.813–0.830],2212/1116130,0.852,0.612
Excluded patients,0.758 [0.731–0.784],343/201534,0.665,0.722
Non-excluded patients,0.834 [0.826–0.842],1867/914320,0.886,0.588
Age < 70,0.824 [0.812–0.835],1133/645537,0.839,0.646
Age 70-80,0.809 [0.791–0.824],602/232581,0.864,0.560
Age 80-85,0.823 [0.794–0.850],231/114817,0.857,0.570
Age > 85,0.827 [0.798–0.856],196/101105,0.878,0.572
Surgical ICU,0.810 [0.798–0.822],1268/555031,0.833,0.613
Non-surgical ICU,0.836 [0.825–0.848],942/560823,0.878,0.611


## Generate Latex Tables

### Die titel ändern, untertitel zu den 3 spalten, formattieren

In [96]:
def escape_for_latex(s):
    return str(s).replace('_', r'\_').replace('%', r'\%') \
                 .replace('#', r'\#') \
                 .replace('<', r'$<$').replace('>', r'$>$') \
                 .replace('[', r'[').replace(']', r']') \
                 .replace('=', r'=')  # = ist meist OK außerhalb Mathemodus

def shorten_count_str(count_str):
    try:
        pos, total = count_str.strip().split('/')
        total = int(total)
        # Teile durch 1000 und runde auf eine Nachkommastelle
        if total < 1000:
            total_str = str(total)
        elif total < 100000:
            total_str = f"{round(total / 1000, 1)}K".replace(".0", "")
        else:
            total_str = f"{round(total / 1000):,}K".replace(",", "")
        return f"{pos}/{total_str}"
    except Exception as e:
        return count_str
    

def format_latex_cell(value):
    """
    Erwarteter Wert: 'auc, pos/total, sens, spec, threshold'
    Gibt zurück:
    AUC: 0.833 & Sens: 0.860 & Spec: 0.633 & [1350/659K]
    """
    try:
        parts = [p.strip() for p in value.split(',')]
        if len(parts) < 4:
            return escape_for_latex(value)

        auc, count_str, sens, spec = parts[:4]
        count_short = shorten_count_str(count_str)
        return f"{auc} & {sens} & {spec} & [{count_short}]"
    except Exception as e:
        return escape_for_latex(value)
    # 0.833 & 1350/659K & 0.86 & 0.63 



mix_biometric_translation = {
    "age_bin": {"1": "< 53.0", "2": "[53.0 - 64.5)", "3": "[64.5 - 76.0)", "4": ">= 76.0"},
    "height_bin": {"1": "< 163.0", "2": "[163.0 - 170.5)", "3": "[170.5 - 178.0)", "4": ">= 178.0"},
    "weight_bin": {"1": "< 65.6", "2": "[65.6 - 79.6)", "3": "[79.6 - 93.6)", "4": ">= 93.6"},
    "bmi_bin": {"1": "< 23.4", "2": "[23.4 - 27.7)", "3": "[27.7 - 32.1)", "4": ">= 32.1"}
}

dem_translation = {
    "gender_bin": {"0": "Male", "1": "Female"},
    "ethnicity_bin": {"0": "White", "1": "Black", "2": "Asian", "3": "Hispanic", "4": "Other"}
}


def translate_bin_names(df):
    def translate_feature(feature):
        match = re.match(r"(\w+_bin) = (\d)", str(feature))
        if match:
            bin_name, bin_value = match.groups()
            if bin_name in mix_biometric_translation:
                label = mix_biometric_translation[bin_name].get(bin_value, bin_value)
                return f"{bin_name[:-4]} {label}"
            elif bin_name in dem_translation:
                label = dem_translation[bin_name].get(bin_value, bin_value)
                return f"{bin_name[:-4]}: {label}"
        return feature

    df['Feature'] = df['Feature'].apply(translate_feature)
    return df


def bold_best_auc_in_row(row: str) -> str:
    blocks = [b.strip() for b in row.split("&")]

    # Identify AUC columns: 1, 5, 9, ... (i.e. every 4th after feature)
    auc_indices = list(range(1, len(blocks), 4))
    aucs = []

    for idx in auc_indices:
        match = re.match(r"([0-9]+\.[0-9]+)", blocks[idx])
        if match:
            aucs.append((idx, float(match.group(1))))

    if not aucs:  # no AUCs found → return unchanged
        return row

    # Find best AUC(s) (support ties)
    best_val = max(val for _, val in aucs)
    best_indices = [idx for idx, val in aucs if val == best_val]

    # Bold all best AUCs
    for idx in best_indices:
        auc_str = re.match(r"([0-9]+\.[0-9]+)", blocks[idx]).group(1)
        blocks[idx] = blocks[idx].replace(
            auc_str, f"\\textbf{{{auc_str}}}", 1
        )

    return " & ".join(blocks)



In [ ]:

df = pd.read_csv("/user/rirg2545/Projects/Hypotension-Project/src/hypotension-individual-threshold/ce_approach/evaluation/subanalysis/results_subgroup_analysis_newest.csv")

df = translate_bin_names(df)
df = df.map(format_latex_cell)


# Als LaTeX-Tabelle ausgeben (ohne Index)
latex_code = "\\begin{scriptsize}\n" + df.to_latex(index=False, escape=False) + "\n\\end{scriptsize}"

print(latex_code)

In [97]:
# 1. Load and translate bin names
df = pd.read_csv("/dss/work/rirg2545/actionable-hypotension/extended_evaluation_review/results/extended_subgroup_analysis.csv")

df = df.map(format_latex_cell)

# 2. Convert DataFrame to LaTeX string
latex_code = df.to_latex(index=False, escape=False)

# 3. Postprocess each LaTeX row to bold the best AUC
processed_lines = []
for line in latex_code.splitlines():
    if "&" in line and not line.strip().startswith("\\"):  # skip headers/commands
        line = bold_best_auc_in_row(line)
    processed_lines.append(line)

# 4. Wrap in LaTeX environment
final_latex = "\\begin{scriptsize}\n" + "\n".join(processed_lines) + "\n\\end{scriptsize}"

print(final_latex)

\begin{scriptsize}
\begin{tabular}{ll}
\toprule
Subgroup & mix \\
\midrule
All patients & \textbf{0.822} [0.813–0.830] & sens=0.852 & spec=0.612 & [2212/1116K] \\
Excluded patients & \textbf{0.758} [0.730–0.786] & sens=0.665 & spec=0.722 & [343/202K] \\
Non-excluded patients & \textbf{0.834} [0.825–0.842] & sens=0.886 & spec=0.588 & [1867/914K] \\
Age $<$ 70 & \textbf{0.824} [0.812–0.835] & sens=0.839 & spec=0.646 & [1133/646K] \\
Age 70-80 & \textbf{0.809} [0.793–0.824] & sens=0.864 & spec=0.560 & [602/233K] \\
Age 80-85 & \textbf{0.823} [0.795–0.851] & sens=0.857 & spec=0.570 & [231/115K] \\
Age $>$ 85 & \textbf{0.827} [0.797–0.856] & sens=0.878 & spec=0.572 & [196/101K] \\
Surgical ICU & \textbf{0.810} [0.798–0.823] & sens=0.833 & spec=0.613 & [1268/555K] \\
Non-surgical ICU & \textbf{0.836} [0.824–0.848] & sens=0.878 & spec=0.611 & [942/561K] \\
Heart surgery & \textbf{0.799} [0.782–0.813] & sens=0.894 & spec=0.499 & [782/254K] \\
No heart surgery & \textbf{0.824} [0.813–0.835] & s